# 03_generate-interactive-webpage.

This notebook loads the cached outputs from `02_run-multi-match-prediction.ipynb` and exports an interactive HTML web-view.

In [13]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

## Configuration

Set the phase and scoreline method to match the cached outputs produced by `02_prediction_matrix.ipynb`.

The exported HTML starts with `Switzerland vs Qatar` selected when that matchup exists in the cached files.

In [14]:
phase = 'group'
top_n_scores = 15
top_n_tips = 15
default_team_a = 'Switzerland'
default_team_b = 'Qatar'
phase_order = [
    'group',
    'round_of_32',
    'round_of_16',
    'quarterfinal',
    'semifinal',
    'third_place',
    'final',
]
phase_labels = {
    'group': 'Group',
    'round_of_32': 'Round of 32',
    'round_of_16': 'Round of 16',
    'quarterfinal': 'Quarterfinal',
    'semifinal': 'Semifinal',
    'third_place': 'Third Place',
    'final': 'Final',
}
web_dir = Path('web')
web_dir.mkdir(parents=True, exist_ok=True)
output_html_path = web_dir / f'interactive-{phase}.html'
path_flag_lut = Path('utils/lut_flag.json')

scoreline_method_suffix = ''
dir_out = Path(f'simulation_results/{phase}')

path_matrix_long = dir_out / f'prediction_matrix_long_{phase}{scoreline_method_suffix}.csv'
detail_cache_paths = {
    'score_distribution': dir_out / f'matchup_score_distribution_{phase}{scoreline_method_suffix}.csv',
    'outcome_distribution': dir_out / f'matchup_outcome_distribution_{phase}{scoreline_method_suffix}.csv',
    'goal_diff_distribution': dir_out / f'matchup_goal_diff_distribution_{phase}{scoreline_method_suffix}.csv',
    'tip_rank': dir_out / f'matchup_tip_rank_{phase}{scoreline_method_suffix}.csv',
}

missing_paths = [path for path in [path_matrix_long, *detail_cache_paths.values()] if not path.exists()]
if missing_paths:
    missing_paths_str = '\n'.join(str(path) for path in missing_paths)
    raise FileNotFoundError(
        'Required cached files are missing. Run 02_prediction_matrix.ipynb first.\n'
        f'{missing_paths_str}'
    )

df_matrix_long = pd.read_csv(path_matrix_long)
df_matchup_details = {
    name: pd.read_csv(path)
    for name, path in detail_cache_paths.items()
}
team_flag_lut = json.loads(path_flag_lut.read_text(encoding='utf-8')) if path_flag_lut.exists() else {}

teams = sorted(df_matrix_long['team_a'].unique())
df_row_win_matrix = (
    df_matrix_long
    .pivot(index='team_a', columns='team_b', values='team_a_win_probability')
    .reindex(index=teams, columns=teams)
)
team_flag_lut = {team: team_flag_lut.get(team, '') for team in teams}

if default_team_a not in teams or default_team_b not in teams or default_team_a == default_team_b:
    default_team_a = teams[0]
    default_team_b = next(team for team in teams if team != default_team_a)

display(Markdown(
    f'Loaded `{path_matrix_long}` and {len(detail_cache_paths)} matchup detail tables. '
    f'The exported HTML will default to `{default_team_a} vs {default_team_b}`.'
))

Loaded `simulation_results\group\prediction_matrix_long_group.csv` and 4 matchup detail tables. The exported HTML will default to `Switzerland vs Qatar`.

In [15]:
def get_matchup_slice(df, team_a, team_b):
    return df[(df['team_a'] == team_a) & (df['team_b'] == team_b)].copy()

def load_matchup_view_data(team_a, team_b, top_n_scores=15, top_n_tips=15):
    summary_row = get_matchup_slice(df_matrix_long, team_a, team_b)
    if summary_row.empty:
        raise ValueError(f'No cached matchup found for {team_a} vs {team_b}.')

    summary_row = summary_row.iloc[0]
    score_distribution = (
        get_matchup_slice(df_matchup_details['score_distribution'], team_a, team_b)
        .sort_values('rank')
        .head(top_n_scores)
    )
    goal_diff_distribution = (
        get_matchup_slice(df_matchup_details['goal_diff_distribution'], team_a, team_b)
        .sort_values('goal_diff')
    )
    tip_rank = (
        get_matchup_slice(df_matchup_details['tip_rank'], team_a, team_b)
        .sort_values('rank')
        .head(top_n_tips)
    )

    return {
        'summary': summary_row,
        'score_distribution': score_distribution,
        'goal_diff_distribution': goal_diff_distribution,
        'tip_rank': tip_rank,
    }

def to_serializable_number(value):
    if pd.isna(value):
        return None
    return float(value)

def build_outcome_payload(summary, team_a, team_b):
    return [
        {
            'outcome': f'{team_a} win',
            'probability_percent': to_serializable_number(summary['team_a_win_probability']),
        },
        {
            'outcome': 'Draw',
            'probability_percent': to_serializable_number(summary['draw_probability']),
        },
        {
            'outcome': f'{team_b} win',
            'probability_percent': to_serializable_number(summary['team_b_win_probability']),
        },
    ]

def build_matchup_payload(team_a, team_b, top_n_scores=15, top_n_tips=15):
    matchup_data = load_matchup_view_data(
        team_a=team_a,
        team_b=team_b,
        top_n_scores=top_n_scores,
        top_n_tips=top_n_tips,
    )
    summary = matchup_data['summary']
    outcomes = build_outcome_payload(summary=summary, team_a=team_a, team_b=team_b)

    return {
        'summary': {
            'team_a': team_a,
            'team_b': team_b,
            'team_a_win_probability': to_serializable_number(summary['team_a_win_probability']),
            'draw_probability': to_serializable_number(summary['draw_probability']),
            'team_b_win_probability': to_serializable_number(summary['team_b_win_probability']),
            'most_common_score': str(summary['most_common_score']),
            'most_common_score_probability': to_serializable_number(summary['most_common_score_probability']),
            'recommended_tip': str(summary['recommended_tip']),
            'recommended_tip_outcome': str(summary['recommended_tip_outcome']),
            'recommended_tip_expected_points': to_serializable_number(summary['recommended_tip_expected_points']),
            'recommended_tip_exact_probability': to_serializable_number(summary['recommended_tip_exact_probability']),
            'avg_goals_a': to_serializable_number(summary['avg_goals_a']),
            'avg_goals_b': to_serializable_number(summary['avg_goals_b']),
            'avg_goal_difference': to_serializable_number(summary['avg_goal_difference']),
        },
        'scores': [
            {
                'score': str(row['score']),
                'probability_percent': to_serializable_number(row['probability_percent']),
            }
            for _, row in matchup_data['score_distribution'].iterrows()
        ],
        'outcomes': outcomes,
        'goal_diffs': [
            {
                'goal_diff': int(row['goal_diff']),
                'probability_percent': to_serializable_number(row['probability_percent']),
            }
            for _, row in matchup_data['goal_diff_distribution'].iterrows()
        ],
        'tips': [
            {
                'tip': str(row['tip']),
                'tip_goals_a': int(row['tip_goals_a']),
                'tip_goals_b': int(row['tip_goals_b']),
                'outcome': str(row['outcome']),
                'expected_points': to_serializable_number(row['expected_points']),
                'exact_score_probability_percent': to_serializable_number(
                    row['exact_score_probability_percent']
                ),
                'rank': int(row['rank']),
            }
            for _, row in matchup_data['tip_rank'].iterrows()
        ],
    }

matchup_payload = {
    f'{team_a}__{team_b}': build_matchup_payload(
        team_a=team_a,
        team_b=team_b,
        top_n_scores=top_n_scores,
        top_n_tips=top_n_tips,
    )
    for team_a in teams
    for team_b in teams
    if team_a != team_b
}

heatmap_z = df_row_win_matrix.where(~df_row_win_matrix.isna(), None).values.tolist()
matrix_z_max = float(df_row_win_matrix.max().max()) if not df_row_win_matrix.empty else 100.0

display(Markdown(
    f'Prepared cached payload for {len(matchup_payload):,} ordered matchups.'
))

Prepared cached payload for 2,256 ordered matchups.

## Export Standalone HTML

This cell writes a self-contained HTML file that can be shared and opened outside Jupyter. The left panel is the clickable win-probability matrix, and the right panel is the matchup-specific 2x2 detail view.

In [ ]:
html_template = """<!DOCTYPE html>
<html lang=\"en\">
<head>
  <meta charset=\"utf-8\">
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">
  <title>World Cup 2026 Predictions for SRF Tippspiel</title>
  <script src=\"https://cdn.plot.ly/plotly-2.35.2.min.js\"></script>
  <style>
    :root {
      --page-bg: #eef3f8;
      --panel-bg: rgba(255, 255, 255, 0.9);
      --panel-border: rgba(148, 163, 184, 0.28);
      --text: #0f172a;
      --muted: #475569;
      --accent: #0f766e;
      --team-a: #1a9850;
      --team-b: #d73027;
      --draw: #6b7280;
      --selection: #4b5563;
      --hover-row: rgba(15, 118, 110, 0.18);
      --hover-col: rgba(15, 23, 42, 0.14);
      --shadow: 0 24px 60px rgba(15, 23, 42, 0.12);
    }
    * { box-sizing: border-box; }
    body {
      margin: 0;
      font-family: Inter, 'Segoe UI', 'Helvetica Neue', Arial, sans-serif;
      color: var(--text);
      background: radial-gradient(circle at top left, #dbeafe 0%, #eef3f8 35%, #f8fafc 100%);
    }
    a { color: var(--accent); }
    .shell {
      max-width: 1360px;
      margin: 0 auto;
      padding: 28px 20px 36px 20px;
    }
    .section-card {
      background: var(--panel-bg);
      border: 1px solid var(--panel-border);
      border-radius: 24px;
      box-shadow: var(--shadow);
      backdrop-filter: blur(16px);
    }
    .detail-section {
      display: grid;
      gap: 18px;
      padding: 22px;
      margin-bottom: 18px;
    }
    .section-intro {
      display: grid;
      gap: 8px;
    }
    .section-intro h3 {
      margin: 0;
      font-size: 24px;
      letter-spacing: -0.03em;
    }
    .section-intro p {
      margin: 0;
      color: var(--muted);
      line-height: 1.6;
    }
    .selection-heading {
      display: grid;
      gap: 10px;
    }
    .selection-heading h2 {
      margin: 0;
      font-size: clamp(28px, 3vw, 42px);
      line-height: 1.02;
      letter-spacing: -0.045em;
      font-weight: 850;
    }
    .selection-heading p {
      margin: 0;
      color: var(--muted);
      font-size: 14px;
      line-height: 1.6;
    }
    .matchup-title {
      display: flex;
      flex-wrap: wrap;
      align-items: center;
      gap: 12px;
      font-size: clamp(24px, 2.2vw, 34px);
      line-height: 1.12;
      letter-spacing: -0.03em;
    }
    .matchup-team {
      display: inline-flex;
      align-items: center;
      gap: 10px;
      font-weight: 800;
      color: var(--text);
    }
    .matchup-versus {
      font-style: italic;
      font-weight: 600;
      color: var(--muted);
    }
    .flag-icon {
      width: 34px;
      height: 25px;
      object-fit: cover;
      border-radius: 4px;
      box-shadow: 0 0 0 1px rgba(148, 163, 184, 0.35);
      background: rgba(255,255,255,0.9);
      vertical-align: middle;
      flex: 0 0 auto;
    }
    .selector-row {
      display: grid;
      grid-template-columns: 1fr 1fr;
      gap: 12px;
    }
    .selector-label {
      display: flex;
      flex-direction: column;
      gap: 6px;
      color: var(--muted);
      font-size: 11px;
      font-weight: 700;
      text-transform: uppercase;
      letter-spacing: 0.12em;
    }
    .selector-label select {
      appearance: none;
      border: 1px solid rgba(100, 116, 139, 0.38);
      border-radius: 14px;
      background: linear-gradient(180deg, rgba(241, 245, 249, 0.98) 0%, rgba(226, 232, 240, 0.92) 100%);
      color: var(--text);
      padding: 12px 14px;
      font-size: 15px;
      font-weight: 600;
      line-height: 1.2;
      box-shadow: inset 0 0 0 1px rgba(255, 255, 255, 0.55), 0 8px 20px rgba(148, 163, 184, 0.12);
    }
    .summary-strip {
      display: grid;
      grid-template-columns: repeat(3, minmax(0, 1fr));
      gap: 12px;
    }
    .summary-card {
      padding: 14px 16px;
      border-radius: 18px;
      background: linear-gradient(180deg, rgba(248, 250, 252, 0.95) 0%, rgba(241, 245, 249, 0.95) 100%);
      border: 1px solid rgba(148, 163, 184, 0.20);
    }
    .summary-card span {
      display: block;
      color: var(--muted);
      font-size: 11px;
      font-weight: 700;
      text-transform: uppercase;
      letter-spacing: 0.12em;
    }
    .summary-card strong {
      display: block;
      margin-top: 8px;
      font-size: clamp(18px, 1.8vw, 26px);
      line-height: 1.15;
      letter-spacing: -0.03em;
    }
    .chart-grid {
      display: grid;
      grid-template-columns: 1fr 1fr;
      gap: 14px;
    }
    .chart-card {
      border-radius: 20px;
      border: 1px solid rgba(148, 163, 184, 0.22);
      background: rgba(255, 255, 255, 0.94);
      overflow: hidden;
      min-height: 340px;
    }
    .detail-chart {
      width: 100%;
      height: 100%;
      min-height: 340px;
    }
    .matrix-section {
      padding: 18px 18px 12px 18px;
      display: grid;
      gap: 12px;
    }
    .matrix-section h3 {
      margin: 0;
      font-size: 24px;
      letter-spacing: -0.03em;
    }
    .matrix-section p {
      margin: 0;
      color: var(--muted);
      line-height: 1.6;
    }
    #matrix-chart {
      width: 100%;
      height: min(84vw, 980px);
      min-height: 520px;
    }
    @media (max-width: 900px) {
      .shell { padding: 18px 14px 28px 14px; }
      .selector-row, .summary-strip, .chart-grid { grid-template-columns: 1fr; }
      #matrix-chart { height: min(112vw, 720px); min-height: 420px; }
    }
  </style>
</head>
<body>
  <div class=\"shell\">
    <section class=\"section-card detail-section\">
      <div class=\"section-intro\">
        <h3>Match outcome and goal probabilities</h3>
      </div>
      <div class=\"selection-heading\" id=\"selection-summary\"></div>
      <div class=\"selector-row\">
        <label class=\"selector-label\">
          Team
          <select id=\"team-a-select\"></select>
        </label>
        <label class=\"selector-label\">
          Opponent
          <select id=\"team-b-select\"></select>
        </label>
      </div>
      <div class=\"summary-strip\" id=\"summary-strip\"></div>
      <div class=\"chart-grid\">
        <div class=\"chart-card\"><div id=\"scores-chart\" class=\"detail-chart\"></div></div>
        <div class=\"chart-card\"><div id=\"outcomes-chart\" class=\"detail-chart\"></div></div>
        <div class=\"chart-card\"><div id=\"tips-chart\" class=\"detail-chart\"></div></div>
        <div class=\"chart-card\"><div id=\"goal-diff-chart\" class=\"detail-chart\"></div></div>
      </div>
    </section>

    <section class=\"section-card matrix-section\">
      <h3>Matchup matrix</h3>
      <p>This matrix allows you to compare win probabilities for each pairing at a glance. Hover over a cell to see the matchup win probability, then click on it to load its detailed score and tipping breakdown above.</p>
      <div id=\"matrix-chart\"></div>
    </section>
  </div>
  <script>
    const teams = __TEAMS__;
    const teamFlags = __TEAM_FLAGS__;
    const heatmapZ = __HEATMAP_Z__;
    const matchupPayload = __MATCHUP_PAYLOAD__;
    const defaultTeamA = __DEFAULT_TEAM_A__;
    const defaultTeamB = __DEFAULT_TEAM_B__;
    const matrixZMax = __MATRIX_Z_MAX__;

    const TEAM_A_COLOR = '#1a9850';
    const TEAM_B_COLOR = '#d73027';
    const DRAW_COLOR = '#6b7280';
    const SELECTION_COLOR = '#4b5563';
    const HOVER_ROW_COLOR = 'rgba(15, 118, 110, 0.18)';
    const HOVER_COL_COLOR = 'rgba(15, 23, 42, 0.14)';
    const PAPER_COLOR = 'rgba(255,255,255,0)';
    const GRID_COLOR = '#e2e8f0';
    const PLOT_FONT = { family: \"Inter, 'Segoe UI', Arial, sans-serif\", size: 12, color: '#0f172a' };
    const MATRIX_SCALE = [[0.0, '#d73027'], [0.5, '#fee08b'], [1.0, '#1a9850']];

    let currentTeamA = defaultTeamA;
    let currentTeamB = defaultTeamB;
    let hoverTeamA = null;
    let hoverTeamB = null;

    function matchupKey(teamA, teamB) {
      return `${teamA}__${teamB}`;
    }

    function getPayload(teamA, teamB) {
      return matchupPayload[matchupKey(teamA, teamB)];
    }

    function getTeamIndex(teamName) {
      return teams.indexOf(teamName);
    }

    function flagUrl(teamName) {
      return teamFlags[teamName] || '';
    }

    function teamLabel(teamName) {
      return teamName;
    }

    function teamLabelWithFlag(teamName) {
      const flag = flagUrl(teamName);
      return flag
        ? `<span class=\"matchup-team\"><img class=\"flag-icon\" src=\"${flag}\" alt=\"${teamName} flag\"> <strong>${teamName}</strong></span>`
        : `<span class=\"matchup-team\"><strong>${teamName}</strong></span>`;
    }

    function scoreColor(score) {
      const [goalsA, goalsB] = score.split('-').map(Number);
      if (goalsA > goalsB) return TEAM_A_COLOR;
      if (goalsB > goalsA) return TEAM_B_COLOR;
      return DRAW_COLOR;
    }

    function outcomeColor(outcome, teamA, teamB) {
      if (outcome === `${teamA} win`) return TEAM_A_COLOR;
      if (outcome === `${teamB} win`) return TEAM_B_COLOR;
      return DRAW_COLOR;
    }

    function goalDiffColor(goalDiff) {
      if (goalDiff > 0) return TEAM_A_COLOR;
      if (goalDiff < 0) return TEAM_B_COLOR;
      return DRAW_COLOR;
    }

    function tipColor(tipObj) {
      if (tipObj.tip_goals_a > tipObj.tip_goals_b) return TEAM_A_COLOR;
      if (tipObj.tip_goals_b > tipObj.tip_goals_a) return TEAM_B_COLOR;
      return DRAW_COLOR;
    }

    function summaryCard(label, value) {
      return `<div class=\"summary-card\"><span>${label}</span><strong>${value}</strong></div>`;
    }

    function selectionRectShape(teamA, teamB) {
      const xIndex = getTeamIndex(teamB);
      const yIndex = getTeamIndex(teamA);
      return {
        type: 'rect',
        xref: 'x',
        yref: 'y',
        x0: xIndex - 0.55,
        x1: xIndex + 0.55,
        y0: yIndex - 0.55,
        y1: yIndex + 0.55,
        fillcolor: 'rgba(0, 0, 0, 0)',
        line: { color: SELECTION_COLOR, width: 3 },
        layer: 'above'
      };
    }

    function hoverBandShapes(teamA, teamB) {
      const xIndex = getTeamIndex(teamB);
      const yIndex = getTeamIndex(teamA);
      if (xIndex < 0 || yIndex < 0) return [];
      return [
        {
          type: 'rect',
          xref: 'x',
          yref: 'y',
          x0: -0.5,
          x1: teams.length - 0.5,
          y0: yIndex - 0.5,
          y1: yIndex + 0.5,
          fillcolor: HOVER_ROW_COLOR,
          line: { width: 0 },
          layer: 'below'
        },
        {
          type: 'rect',
          xref: 'x',
          yref: 'y',
          x0: xIndex - 0.5,
          x1: xIndex + 0.5,
          y0: -0.5,
          y1: teams.length - 0.5,
          fillcolor: HOVER_COL_COLOR,
          line: { width: 0 },
          layer: 'below'
        }
      ];
    }

    function matrixShapes() {
      const shapes = [];
      if (hoverTeamA && hoverTeamB) {
        shapes.push(...hoverBandShapes(hoverTeamA, hoverTeamB));
      }
      shapes.push(selectionRectShape(currentTeamA, currentTeamB));
      return shapes;
    }

    function syncMatrixColorbar() {
      const graph = document.getElementById('matrix-chart');
      if (!graph) return;

      const isVerticalView = window.matchMedia('(max-width: 900px)').matches;

      if (isVerticalView) {
        Plotly.relayout(graph, {
          margin: { l: 72, r: 24, t: 18, b: 118 }
        });

        Plotly.restyle(graph, {
          'colorbar.orientation': ['h'],
          'colorbar.x': [0.5],
          'colorbar.xanchor': ['center'],
          'colorbar.y': [-0.24],
          'colorbar.yanchor': ['top'],
          'colorbar.lenmode': ['fraction'],
          'colorbar.len': [0.78],
          'colorbar.thickness': [12]
        }, [0]);
      } else {
        Plotly.relayout(graph, {
          margin: { l: 92, r: 64, t: 18, b: 92 }
        });

        Plotly.restyle(graph, {
          'colorbar.orientation': ['v'],
          'colorbar.x': [1.02],
          'colorbar.xanchor': ['left'],
          'colorbar.y': [0.5],
          'colorbar.yanchor': ['middle'],
          'colorbar.lenmode': ['fraction'],
          'colorbar.len': [0.5],
          'colorbar.thickness': [14]
        }, [0]);
      }
    }

    function refreshMatrixShapes() {
      Plotly.relayout('matrix-chart', { shapes: matrixShapes() });
    }

    function populateSelectors() {
      const teamASelect = document.getElementById('team-a-select');
      const teamBSelect = document.getElementById('team-b-select');
      teamASelect.innerHTML = teams.map(team => {
        const selected = team === currentTeamA ? ' selected' : '';
        return `<option value=\"${team}\"${selected}>${teamLabel(team)}</option>`;
      }).join('');
      teamBSelect.innerHTML = teams.map(team => {
        const selected = team === currentTeamB ? ' selected' : '';
        return `<option value=\"${team}\"${selected}>${teamLabel(team)}</option>`;
      }).join('');
    }

    function ensureDistinctSelection(changedSide) {
      if (currentTeamA !== currentTeamB) return;
      if (changedSide === 'a') {
        currentTeamB = teams.find(team => team !== currentTeamA) || currentTeamB;
      } else {
        currentTeamA = teams.find(team => team !== currentTeamB) || currentTeamA;
      }
    }

    const matrixData = [{
      type: 'heatmap',
      z: heatmapZ,
      x: teams,
      y: teams,
      colorscale: MATRIX_SCALE,
      zmin: 0,
      zmax: matrixZMax,
      xgap: 1.5,
      ygap: 1.5,
      hoverongaps: false,
      colorbar: {
        title: 'Win probability (%)',
        tickfont: PLOT_FONT,
        titlefont: PLOT_FONT,
        orientation: 'v',
        lenmode: 'fraction',
        len: 0.5,
        x: 1.02,
        xanchor: 'left',
        y: 0.5,
        yanchor: 'middle',
        thickness: 14
      },
      hovertemplate: 'Row team: %{y}<br>Opponent: %{x}<br>Win probability: %{z:.1f}%<extra></extra>'
    }];

    const matrixLayout = {
      margin: { l: 92, r: 64, t: 18, b: 92 },
      paper_bgcolor: PAPER_COLOR,
      plot_bgcolor: PAPER_COLOR,
      font: PLOT_FONT,
      uirevision: 'matrix-static',
      shapes: matrixShapes(),
      xaxis: {
        type: 'category',
        categoryorder: 'array',
        categoryarray: teams,
        tickangle: 45,
        side: 'bottom',
        title: '',
        showgrid: false,
        zeroline: false,
        fixedrange: true,
        range: [-0.5, teams.length - 0.5],
        constrain: 'domain'
      },
      yaxis: {
        type: 'category',
        categoryorder: 'array',
        categoryarray: teams,
        title: '',
        autorange: 'reversed',
        showgrid: false,
        zeroline: false,
        fixedrange: true,
        range: [teams.length - 0.5, -0.5],
        scaleanchor: 'x',
        scaleratio: 1,
        constrain: 'domain'
      }
    };

    Plotly.newPlot('matrix-chart', matrixData, matrixLayout, { responsive: true, displayModeBar: false }).then(syncMatrixColorbar);

    let resizeTimer = null;
    window.addEventListener('resize', () => {
      if (resizeTimer) window.clearTimeout(resizeTimer);
      resizeTimer = window.setTimeout(syncMatrixColorbar, 80);
    });

    function renderSummary(teamA, teamB, payload) {
      document.getElementById('selection-summary').innerHTML = `
        <div class=\"matchup-title\">
          ${teamLabelWithFlag(teamA)}
          <span class=\"matchup-versus\">vs</span>
          ${teamLabelWithFlag(teamB)}
        </div>
        <p>Outcome predictions for the selected pairing are based on a Monte Carlo simulation involving 10,000 repeated simulations with the above models and added perturbations. Select teams with the two selectors below to change the matchup. The charts below show the most likely scorelines, the distribution of wins, draws and losses, the spread of goal differences, and the SRF Tippspiel tips with the highest expected return.</p>
      `;

      document.getElementById('summary-strip').innerHTML = [
        summaryCard('Recommended tip score (SRF Tippspiel)', payload.summary.recommended_tip),
        summaryCard('Expected points (SRF Tippspiel)', payload.summary.recommended_tip_expected_points.toFixed(2)),
        summaryCard('Most likely score', `${payload.summary.most_common_score} (${payload.summary.most_common_score_probability.toFixed(1)}%)`)
      ].join('');
    }

    function baseBarLayout(titleText) {
      return {
        title: { text: titleText, x: 0.04, xanchor: 'left', font: { family: PLOT_FONT.family, size: 16, color: '#0f172a' } },
        margin: { l: 86, r: 22, t: 54, b: 44 },
        paper_bgcolor: PAPER_COLOR,
        plot_bgcolor: PAPER_COLOR,
        font: PLOT_FONT,
        showlegend: false
      };
    }

    function renderDetail(teamA, teamB, syncSelectors = true) {
      const payload = getPayload(teamA, teamB);
      if (!payload) return;

      currentTeamA = teamA;
      currentTeamB = teamB;
      refreshMatrixShapes();
      renderSummary(teamA, teamB, payload);
      if (syncSelectors) populateSelectors();

      Plotly.react('scores-chart', [{
        type: 'bar',
        orientation: 'h',
        x: payload.scores.map(item => item.probability_percent),
        y: payload.scores.map(item => item.score),
        marker: { color: payload.scores.map(item => scoreColor(item.score)) },
        hovertemplate: '%{y}: %{x:.1f}%<extra></extra>'
      }], Object.assign(baseBarLayout('Most Common Scores'), {
        xaxis: { title: 'Probability (%)', gridcolor: GRID_COLOR, zeroline: false },
        yaxis: { title: '', autorange: 'reversed', zeroline: false }
      }), { responsive: true, displayModeBar: false });

      Plotly.react('outcomes-chart', [{
        type: 'bar',
        orientation: 'h',
        x: payload.outcomes.map(item => item.probability_percent),
        y: payload.outcomes.map(item => item.outcome),
        marker: { color: payload.outcomes.map(item => outcomeColor(item.outcome, teamA, teamB)) },
        hovertemplate: '%{y}: %{x:.1f}%<extra></extra>'
      }], Object.assign(baseBarLayout('Outcome Distribution'), {
        xaxis: { title: 'Probability (%)', gridcolor: GRID_COLOR, zeroline: false },
        yaxis: { title: '', autorange: 'reversed', zeroline: false }
      }), { responsive: true, displayModeBar: false });

      Plotly.react('tips-chart', [{
        type: 'bar',
        orientation: 'h',
        x: payload.tips.map(item => item.expected_points),
        y: payload.tips.map(item => item.tip),
        marker: { color: payload.tips.map(item => tipColor(item)) },
        hovertemplate: 'Tip %{y}<br>Expected points: %{x:.2f}<extra></extra>'
      }], Object.assign(baseBarLayout('Best Tips by Expected Points (SRF Tippspiel)'), {
        xaxis: { title: 'Expected Points', gridcolor: GRID_COLOR, zeroline: false },
        yaxis: { title: '', autorange: 'reversed', zeroline: false }
      }), { responsive: true, displayModeBar: false });

      Plotly.react('goal-diff-chart', [{
        type: 'bar',
        x: payload.goal_diffs.map(item => item.goal_diff),
        y: payload.goal_diffs.map(item => item.probability_percent),
        marker: { color: payload.goal_diffs.map(item => goalDiffColor(item.goal_diff)) },
        hovertemplate: 'Goal difference %{x}<br>Probability: %{y:.1f}%<extra></extra>'
      }], Object.assign(baseBarLayout('Goal-Difference Distribution'), {
        xaxis: { title: `Goal Difference (${teamA} - ${teamB})`, zeroline: false },
        yaxis: { title: 'Probability (%)', gridcolor: GRID_COLOR, zeroline: false }
      }), { responsive: true, displayModeBar: false });
    }

    document.getElementById('team-a-select').addEventListener('change', event => {
      currentTeamA = event.target.value;
      ensureDistinctSelection('a');
      renderDetail(currentTeamA, currentTeamB);
    });

    document.getElementById('team-b-select').addEventListener('change', event => {
      currentTeamB = event.target.value;
      ensureDistinctSelection('b');
      renderDetail(currentTeamA, currentTeamB);
    });

    const matrixChart = document.getElementById('matrix-chart');
    matrixChart.on('plotly_click', eventData => {
      if (!eventData || !eventData.points || eventData.points.length === 0) return;
      const point = eventData.points[0];
      const teamB = point.x;
      const teamA = point.y;
      if (teamA === teamB) return;
      renderDetail(teamA, teamB);
    });

    matrixChart.on('plotly_hover', eventData => {
      if (!eventData || !eventData.points || eventData.points.length === 0) return;
      const point = eventData.points[0];
      if (point.x === point.y) return;
      hoverTeamA = point.y;
      hoverTeamB = point.x;
      refreshMatrixShapes();
    });

    matrixChart.on('plotly_unhover', () => {
      hoverTeamA = null;
      hoverTeamB = null;
      refreshMatrixShapes();
    });

    populateSelectors();
    renderDetail(defaultTeamA, defaultTeamB, false);
  </script>
</body>
</html>
"""

html_output = (
    html_template
    .replace('__TEAMS__', json.dumps(teams))
    .replace('__TEAM_FLAGS__', json.dumps(team_flag_lut))
    .replace('__HEATMAP_Z__', json.dumps(heatmap_z))
    .replace('__MATCHUP_PAYLOAD__', json.dumps(matchup_payload))
    .replace('__MATRIX_Z_MAX__', json.dumps(matrix_z_max))
    .replace('__DEFAULT_TEAM_A__', json.dumps(default_team_a))
    .replace('__DEFAULT_TEAM_B__', json.dumps(default_team_b))
)

output_html_path.write_text(html_output, encoding='utf-8')

available_phase_paths = {
    phase_name: web_dir / f'interactive-{phase_name}.html'
    for phase_name in phase_order
    if (web_dir / f'interactive-{phase_name}.html').exists()
}
default_index_phase = phase if phase in available_phase_paths else next(
    (phase_name for phase_name in phase_order if phase_name in available_phase_paths),
    None,
)

phase_tabs_html = '\n'.join(
    (
        f'<button class="phase-tab{' is-active' if phase_name == default_index_phase else ''}{' is-disabled' if phase_name not in available_phase_paths else ''}" '
        f'data-phase="{phase_name}" {'disabled' if phase_name not in available_phase_paths else ''}>'
        f'{phase_labels.get(phase_name, phase_name.title())}</button>'
    )
    for phase_name in phase_order
)

index_viewer_html = (
    f'<iframe id="phase-frame" src="web/interactive-{default_index_phase}.html"></iframe>'
    if default_index_phase
    else '<div class="empty-state">No interactive phase pages are available yet. Export a phase notebook to create a page in the web folder.</div>'
)

index_template = f"""<!DOCTYPE html>
<html lang=\"en\">
<head>
  <meta charset=\"utf-8\">
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">
  <title>World Cup 2026 Predictions for SRF Tippspiel</title>
  <style>
    :root {{
      --page-bg: #eef3f8;
      --panel-bg: rgba(255, 255, 255, 0.9);
      --panel-border: rgba(148, 163, 184, 0.28);
      --text: #0f172a;
      --muted: #475569;
      --accent: #0f766e;
      --shadow: 0 24px 60px rgba(15, 23, 42, 0.12);
    }}
    * {{ box-sizing: border-box; }}
    body {{
      margin: 0;
      font-family: Inter, 'Segoe UI', 'Helvetica Neue', Arial, sans-serif;
      color: var(--text);
      background: radial-gradient(circle at top left, #dbeafe 0%, #eef3f8 35%, #f8fafc 100%);
    }}
    .shell {{
      max-width: 1360px;
      margin: 0 auto;
      padding: 28px 20px 36px 20px;
    }}
    .header {{
      display: grid;
      gap: 12px;
      margin-bottom: 18px;
    }}
    .header h1 {{
      margin: 0;
      font-size: clamp(32px, 4vw, 52px);
      line-height: 1.02;
      letter-spacing: -0.05em;
      font-weight: 850;
    }}
    .header p {{
      margin: 0;
      max-width: 980px;
      color: var(--muted);
      line-height: 1.65;
      font-size: 15px;
    }}
    .badges {{
      display: flex;
      flex-wrap: wrap;
      gap: 10px;
      align-items: center;
    }}
    .badges img {{ height: 28px; display: block; }}
    .tabs {{
      display: flex;
      flex-wrap: wrap;
      gap: 10px;
      margin-bottom: 18px;
    }}
    .phase-tab {{
      border: 1px solid rgba(15, 118, 110, 0.18);
      border-radius: 999px;
      padding: 10px 16px;
      background: rgba(255, 255, 255, 0.9);
      color: var(--text);
      font-size: 14px;
      font-weight: 700;
      cursor: pointer;
    }}
    .phase-tab.is-active {{
      background: rgba(15, 118, 110, 0.14);
      color: var(--accent);
      border-color: rgba(15, 118, 110, 0.35);
    }}
    .phase-tab.is-disabled {{
      background: rgba(226, 232, 240, 0.8);
      color: #94a3b8;
      border-color: rgba(203, 213, 225, 0.9);
      cursor: not-allowed;
    }}
    .viewer {{
      background: var(--panel-bg);
      border: 1px solid var(--panel-border);
      border-radius: 24px;
      box-shadow: var(--shadow);
      overflow: hidden;
      min-height: calc(100vh - 260px);
    }}
    .viewer iframe {{
      width: 100%;
      height: calc(100vh - 260px);
      min-height: 900px;
      border: 0;
      display: block;
      background: transparent;
    }}
    .empty-state {{ padding: 48px 28px; color: var(--muted); line-height: 1.6; }}
    @media (max-width: 900px) {{
      .shell {{ padding: 18px 14px 28px 14px; }}
      .badges img {{ height: 24px; }}
      .viewer iframe {{ min-height: 760px; }}
    }}
  </style>
</head>
<body>
  <div class=\"shell\">
    <div class=\"header\">
      <h1>World Cup 2026 Predictions for SRF Tippspiel</h1>
      <p>This project gives machine-learning based predictions for the 2026 Football World Cup. The predictions are based on a two-stage model. It reuses the original outcome model and data pipeline of <a href="https://github.com/javierruanohdez/world-cup-2026-prediction" target="_blank" rel="noreferrer">Javier Ruano's World Cup 2026 prediction model</a> to predict win-draw-loss probabilities. It then adds per-match score simulations and optimal tip recommendations for the SRF Sport FIFA World Cup 2026 Tippspiel. Predictions start with a supervised <a href="https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html" target="_blank" rel="noreferrer">GradientBoostingClassifier</a> that models each match as a win-draw-loss classification problem. Those outcome probabilities feed a second-stage goal layer that uses two trained <a href="https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html" target="_blank" rel="noreferrer">GradientBoostingRegressor</a> models for total goals and goal difference. Their predictions are converted into expected home and away goals, and final scoreline probabilities are generated from a Poisson-based sampler.</p>
      <div class=\"badges\">
        <a href=\"https://github.com/mibrechb/world-cup-2026-prediction\" target=\"_blank\" rel=\"noreferrer\"><img alt=\"GitHub repo badge\" src=\"https://img.shields.io/badge/GitHub-mibrechb%2Fworld--cup--2026--prediction-111827?logo=github&logoColor=white\"></a>
      </div>
    </div>
    <div class=\"tabs\">
      {phase_tabs_html}
    </div>
    <div class=\"viewer\">
      {index_viewer_html}
    </div>
  </div>
  <script>
    const tabs = Array.from(document.querySelectorAll('.phase-tab:not(.is-disabled)'));
    const frame = document.getElementById('phase-frame');
    tabs.forEach(tab => {{
      tab.addEventListener('click', () => {{
        const phaseName = tab.dataset.phase;
        tabs.forEach(other => other.classList.remove('is-active'));
        tab.classList.add('is-active');
        if (frame) frame.src = `web/interactive-${{phaseName}}.html`;
      }});
    }});
  </script>
</body>
</html>
"""

Path('index.html').write_text(index_template, encoding='utf-8')

display(Markdown(
    f'Exported standalone interactive dashboard to `{output_html_path}` and regenerated `index.html`.'
))

Exported standalone interactive dashboard to `web\interactive-group.html` and regenerated `index.html`.